<a href="https://colab.research.google.com/github/CassieMarie0728/colab-notebooks/blob/main/RVC_Dataset_Preprocessor_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RVC Dataset Preprocessor — Colab Notebook

This notebook builds a clean, organized, RVC-style dataset from raw vocal files.

It takes you from source audio to:
- cleaned WAVs
- silence-sliced clips
- `raw/`
- `sliced/`
- `logs/0_gt_wavs/`
- `filelist.txt`
- CSV reports
- a downloadable ZIP

## What it does
- Upload one or more vocal audio files
- Convert to mono WAV
- Resample to your target sample rate
- Normalize loudness
- Optionally run light denoise
- Slice audio into training-friendly chunks
- Reject clips that are too short or too quiet
- Build an RVC-style folder layout
- Generate `filelist.txt`
- Export the dataset as a ZIP

## What it does not do
- HuBERT feature extraction
- F0 extraction
- RVC training
- stem separation

This is for dataset prep, not the full training pipeline.

## 1) Install dependencies

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!pip -q install librosa soundfile pydub tqdm noisereduce numpy scipy pandas matplotlib

## 2) Imports

In [ ]:
import os
import re
import json
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from pydub import AudioSegment, silence
from google.colab import files

warnings.filterwarnings("ignore")

## 3) Configuration

Recommended defaults:
- `TARGET_SR = 40000`
- `MIN_CLIP_MS = 2500`
- `MAX_CLIP_MS = 12000`
- `SILENCE_THRESH_DBFS = -40`
- `KEEP_SILENCE_MS = 150`

If clips get shredded too hard:
- lower `SILENCE_THRESH_DBFS` to `-45` or `-50`
- reduce `MIN_SILENCE_LEN_MS`

If the notebook barely slices anything:
- raise `SILENCE_THRESH_DBFS` to `-35`
- increase `MIN_SILENCE_LEN_MS`

In [ ]:
DATASET_NAME = "rvc_dataset"
SPEAKER_ID = 0

TARGET_SR = 40000
MONO = True
NORMALIZE_AUDIO = True
PEAK_TARGET = 0.95
USE_DENOISE = False
TRIM_LEADING_TRAILING_SILENCE = True

MIN_SILENCE_LEN_MS = 400
SILENCE_THRESH_DBFS = -40
KEEP_SILENCE_MS = 150

MIN_CLIP_MS = 2500
MAX_CLIP_MS = 12000
MIN_RMS = 0.005

ALLOWED_EXTS = {".wav", ".mp3", ".flac", ".m4a", ".ogg", ".aac", ".opus"}

BASE_DIR = Path("/content") / DATASET_NAME
RAW_DIR = BASE_DIR / "raw"
SLICED_DIR = BASE_DIR / "sliced"
LOGS_DIR = BASE_DIR / "logs"
GT_WAVS_DIR = LOGS_DIR / "0_gt_wavs"
REPORTS_DIR = BASE_DIR / "reports"

for d in [BASE_DIR, RAW_DIR, SLICED_DIR, LOGS_DIR, GT_WAVS_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Dataset root: {BASE_DIR}")

## 4) Upload your source audio

In [ ]:
uploaded = files.upload()

print(f"Uploaded {len(uploaded)} file(s).")

saved_files = []
for name, data in uploaded.items():
    ext = Path(name).suffix.lower()
    if ext not in ALLOWED_EXTS:
        print(f"Skipping unsupported file: {name}")
        continue
    out_path = RAW_DIR / name
    with open(out_path, "wb") as f:
        f.write(data)
    saved_files.append(out_path)

for p in saved_files:
    print(" -", p)

## 5) Inspect uploaded files

In [ ]:
raw_files = sorted([p for p in RAW_DIR.iterdir() if p.suffix.lower() in ALLOWED_EXTS])

rows = []
for p in raw_files:
    try:
        info = sf.info(str(p))
        rows.append({
            "file": p.name,
            "samplerate": info.samplerate,
            "channels": info.channels,
            "frames": info.frames,
            "duration_sec": round(info.frames / info.samplerate, 2),
            "format": info.format,
            "subtype": info.subtype,
        })
    except Exception:
        rows.append({
            "file": p.name,
            "samplerate": None,
            "channels": None,
            "frames": None,
            "duration_sec": None,
            "format": "unknown",
            "subtype": "unknown",
        })

df_raw = pd.DataFrame(rows)
df_raw

## 6) Helper functions

In [ ]:
import noisereduce as nr

def safe_stem(name: str) -> str:
    stem = Path(name).stem
    stem = re.sub(r"[^a-zA-Z0-9_\-]+", "_", stem)
    stem = re.sub(r"_+", "_", stem).strip("_")
    return stem or "audio"

def load_audio_any(path, sr=40000, mono=True):
    y, _ = librosa.load(path, sr=sr, mono=mono)
    return y.astype(np.float32)

def trim_silence(y, top_db=35):
    yt, _ = librosa.effects.trim(y, top_db=top_db)
    return yt.astype(np.float32)

def peak_normalize(y, peak_target=0.95):
    peak = np.max(np.abs(y)) if len(y) else 0
    if peak <= 0:
        return y
    return (y / peak * peak_target).astype(np.float32)

def rms(y):
    if len(y) == 0:
        return 0.0
    return float(np.sqrt(np.mean(np.square(y))))

def denoise_audio(y, sr):
    return nr.reduce_noise(y=y, sr=sr, stationary=True, prop_decrease=0.75).astype(np.float32)

def save_wav(path, y, sr):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(str(path), y, sr, subtype="PCM_16")

def wav_to_audiosegment(path):
    return AudioSegment.from_wav(path)

def split_long_segment(seg, max_len_ms):
    if len(seg) <= max_len_ms:
        return [seg]
    pieces = []
    start = 0
    while start < len(seg):
        end = min(start + max_len_ms, len(seg))
        pieces.append(seg[start:end])
        start = end
    return pieces

def clip_is_valid(y, min_rms=0.005):
    return len(y) > 0 and rms(y) >= min_rms and np.max(np.abs(y)) > 1e-4

## 7) Convert, clean, and normalize source files

In [ ]:
CLEANED_RAW_DIR = RAW_DIR / "cleaned_wavs"
CLEANED_RAW_DIR.mkdir(parents=True, exist_ok=True)

cleaned_rows = []

for src in tqdm(raw_files, desc="Cleaning source audio"):
    try:
        y = load_audio_any(src, sr=TARGET_SR, mono=MONO)

        if TRIM_LEADING_TRAILING_SILENCE:
            y = trim_silence(y, top_db=35)

        if USE_DENOISE and len(y) > 0:
            y = denoise_audio(y, TARGET_SR)

        if NORMALIZE_AUDIO and len(y) > 0:
            y = peak_normalize(y, peak_target=PEAK_TARGET)

        out_name = f"{safe_stem(src.name)}.wav"
        out_path = CLEANED_RAW_DIR / out_name
        save_wav(out_path, y, TARGET_SR)

        cleaned_rows.append({
            "source_file": src.name,
            "cleaned_file": out_name,
            "duration_sec": round(len(y) / TARGET_SR, 2),
            "rms": round(rms(y), 6),
            "status": "ok"
        })
    except Exception as e:
        cleaned_rows.append({
            "source_file": src.name,
            "cleaned_file": None,
            "duration_sec": None,
            "rms": None,
            "status": f"error: {e}"
        })

df_cleaned = pd.DataFrame(cleaned_rows)
df_cleaned

## 8) Slice cleaned audio into training clips

In [ ]:
cleaned_wavs = sorted(CLEANED_RAW_DIR.glob("*.wav"))

slice_rows = []

for cleaned_path in tqdm(cleaned_wavs, desc="Slicing audio"):
    base = safe_stem(cleaned_path.name)

    try:
        seg = wav_to_audiosegment(cleaned_path)
        if seg.channels != 1:
            seg = seg.set_channels(1)
        if seg.frame_rate != TARGET_SR:
            seg = seg.set_frame_rate(TARGET_SR)

        chunks = silence.split_on_silence(
            seg,
            min_silence_len=MIN_SILENCE_LEN_MS,
            silence_thresh=SILENCE_THRESH_DBFS,
            keep_silence=KEEP_SILENCE_MS,
        )

        if not chunks:
            chunks = [seg]

        clip_index = 0
        kept = 0
        rejected = 0

        for chunk in chunks:
            for piece in split_long_segment(chunk, MAX_CLIP_MS):
                if len(piece) < MIN_CLIP_MS:
                    rejected += 1
                    continue

                clip_name = f"{base}_{clip_index:04d}.wav"
                temp_path = SLICED_DIR / clip_name
                piece.export(temp_path, format="wav")

                y, _ = librosa.load(temp_path, sr=TARGET_SR, mono=True)

                if not clip_is_valid(y, min_rms=MIN_RMS):
                    temp_path.unlink(missing_ok=True)
                    rejected += 1
                    clip_index += 1
                    continue

                if NORMALIZE_AUDIO:
                    y = peak_normalize(y, peak_target=PEAK_TARGET)
                    save_wav(temp_path, y, TARGET_SR)

                gt_path = GT_WAVS_DIR / clip_name
                shutil.copy2(temp_path, gt_path)

                kept += 1
                clip_index += 1

        slice_rows.append({
            "file": cleaned_path.name,
            "clips_kept": kept,
            "clips_rejected": rejected,
            "status": "ok"
        })

    except Exception as e:
        slice_rows.append({
            "file": cleaned_path.name,
            "clips_kept": 0,
            "clips_rejected": 0,
            "status": f"error: {e}"
        })

df_slices = pd.DataFrame(slice_rows)
df_slices

## 9) Review the generated clips

In [ ]:
generated_clips = sorted(GT_WAVS_DIR.glob("*.wav"))

rows = []
for p in generated_clips:
    info = sf.info(str(p))
    y, _ = librosa.load(p, sr=TARGET_SR, mono=True)
    rows.append({
        "clip": p.name,
        "duration_sec": round(info.frames / info.samplerate, 2),
        "rms": round(rms(y), 6),
        "peak": round(float(np.max(np.abs(y))) if len(y) else 0.0, 6),
    })

df_clips = pd.DataFrame(rows).sort_values(["duration_sec", "clip"], ascending=[False, True])
print(f"Total generated clips: {len(df_clips)}")
df_clips.head(25)

## 10) Quick visualization

In [ ]:
if len(df_clips) > 0:
    plt.figure(figsize=(12, 4))
    plt.hist(df_clips["duration_sec"], bins=25)
    plt.title("Clip Duration Distribution")
    plt.xlabel("Seconds")
    plt.ylabel("Count")
    plt.show()

    plt.figure(figsize=(12, 4))
    plt.hist(df_clips["rms"], bins=25)
    plt.title("Clip RMS Distribution")
    plt.xlabel("RMS")
    plt.ylabel("Count")
    plt.show()
else:
    print("No clips found. Adjust your slicing settings and rerun.")

## 11) Build `filelist.txt`

In [ ]:
filelist_path = BASE_DIR / "filelist.txt"

with open(filelist_path, "w", encoding="utf-8") as f:
    for wav_path in sorted(GT_WAVS_DIR.glob("*.wav")):
        rel = wav_path.relative_to(BASE_DIR).as_posix()
        f.write(f"{rel}|{SPEAKER_ID}\n")

print(f"Wrote: {filelist_path}")

with open(filelist_path, "r", encoding="utf-8") as f:
    preview = "".join(f.readlines()[:10])

print("\nPreview:\n")
print(preview if preview else "[filelist is empty]")

## 12) Save reports

In [ ]:
df_raw.to_csv(REPORTS_DIR / "raw_files_report.csv", index=False)
df_cleaned.to_csv(REPORTS_DIR / "cleaned_files_report.csv", index=False)
df_slices.to_csv(REPORTS_DIR / "slice_report.csv", index=False)
df_clips.to_csv(REPORTS_DIR / "generated_clips_report.csv", index=False)

summary = {
    "dataset_name": DATASET_NAME,
    "speaker_id": SPEAKER_ID,
    "target_sr": TARGET_SR,
    "total_source_files": int(len(df_raw)),
    "total_cleaned_files": int((df_cleaned["status"] == "ok").sum()) if len(df_cleaned) else 0,
    "total_generated_clips": int(len(df_clips)),
    "settings": {
        "mono": MONO,
        "normalize_audio": NORMALIZE_AUDIO,
        "peak_target": PEAK_TARGET,
        "use_denoise": USE_DENOISE,
        "trim_leading_trailing_silence": TRIM_LEADING_TRAILING_SILENCE,
        "min_silence_len_ms": MIN_SILENCE_LEN_MS,
        "silence_thresh_dbfs": SILENCE_THRESH_DBFS,
        "keep_silence_ms": KEEP_SILENCE_MS,
        "min_clip_ms": MIN_CLIP_MS,
        "max_clip_ms": MAX_CLIP_MS,
        "min_rms": MIN_RMS,
    }
}

with open(REPORTS_DIR / "summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Saved report files to:", REPORTS_DIR)

## 13) Package the dataset as a ZIP

In [ ]:
zip_base = f"/content/{DATASET_NAME}"
zip_path = shutil.make_archive(zip_base, "zip", root_dir=BASE_DIR)
print("Created ZIP:", zip_path)

## 14) Download the ZIP

In [ ]:
files.download(f"/content/{DATASET_NAME}.zip")

## Folder layout produced

```text
rvc_dataset/
├── raw/
│   ├── original uploads
│   └── cleaned_wavs/
├── sliced/
├── logs/
│   └── 0_gt_wavs/
├── reports/
├── filelist.txt
```

## Important reality check
This notebook prepares the dataset structure and training clips.

It does not generate:
- `2a_f0/`
- `2b-f0nsf/`
- `3_feature256/`

Those are created later in the actual RVC preprocessing / feature extraction pipeline.